# 04 -- Feature Engineering
**Purpose:** Transform raw columns into ML-ready features that capture fraud signals the model can learn from.

**Input:** `data/processed/transactions_clean.parquet`  
**Output:** `data/processed/feature_matrix.parquet` | `data/processed/target.parquet`

## Why Feature Engineering?
`amount_usd = 74000` tells a model a number. `amount_zscore_90d = 4.2` tells it this
transaction is 4.2 standard deviations above the customer's 90-day average — a strong anomaly.
Feature engineering is where domain knowledge becomes model signal.

## Section 1 — Setup

In [1]:
import pandas as pd
import numpy as np
import os

# Load clean transactions from Notebook 03 output
df = pd.read_parquet('../data/processed/transactions_clean.parquet')
customers = pd.read_csv('../data/raw/customers.csv')
accounts  = pd.read_csv('../data/raw/accounts.csv')

print('Transactions shape:', df.shape)
print('Customers shape:   ', customers.shape)
print('Accounts shape:    ', accounts.shape)
print('Fraud rate:        ', df['is_fraud'].mean().round(5))

Transactions shape: (50800, 63)
Customers shape:    (500, 31)
Accounts shape:     (780, 13)
Fraud rate:         0.00492


## Section 2 — Transaction Features (7)
These features require only the transactions table — no join needed.

In [2]:
# Feature 1: log_amount_usd
# Log transformation compresses the long right tail of amount values.
# +1 avoids log(0) error for zero-amount transactions.
df['log_amount_usd'] = np.log1p(df['amount_usd'])

# Features 2 & 3: submission_hour, submission_day
# Fraud has a time signature — legitimate payments cluster in business hours on weekdays.
df['submission_hour'] = df['timestamp'].dt.hour
df['submission_day']  = df['timestamp'].dt.dayofweek  # 0=Monday, 6=Sunday

# Feature 4: is_near_threshold
# Structuring: sending amounts just below USD 10,000 to avoid AML reporting.
THRESHOLD = 10000
BAND = 0.10
df['is_near_threshold'] = (
    (df['amount_usd'] >= THRESHOLD * (1 - BAND)) &
    (df['amount_usd'] <  THRESHOLD)
).astype(int)

# Feature 5: is_off_hours
# Transactions before 9am or after 6pm are statistically unusual for business payments.
df['is_off_hours'] = (
    (df['submission_hour'] < 9) | (df['submission_hour'] > 18)
).astype(int)

# Feature 6: is_weekend
# Saturday=5, Sunday=6
df['is_weekend'] = (df['submission_day'] >= 5).astype(int)

# Feature 7: rail_corridor_mismatch
# Using a country-specific payment rail for the wrong destination country is a red flag.
# Fedwire/CHIPS = US only | CHAPS = UK only | SEPA = Europe only | SWIFT = universal
FEDWIRE_CHIPS_COUNTRIES = {'US'}
CHAPS_COUNTRIES         = {'UK'}
SEPA_COUNTRIES          = {'Germany', 'France', 'Netherlands', 'Italy', 'Spain',
                            'Belgium', 'Austria', 'Sweden', 'Denmark', 'Finland'}

def is_mismatch(row):
    rail    = row['payment_rail']
    country = row['beneficiary_country']
    if rail in ('Fedwire', 'CHIPS') and country not in FEDWIRE_CHIPS_COUNTRIES:
        return 1
    if rail == 'CHAPS' and country not in CHAPS_COUNTRIES:
        return 1
    if rail == 'SEPA' and country not in SEPA_COUNTRIES:
        return 1
    return 0

df['rail_corridor_mismatch'] = df.apply(is_mismatch, axis=1)

print('Transaction features built. Shape:', df.shape)

Transaction features built. Shape: (50800, 70)


## Section 3 — Customer and Account Features (8)
Most customer and account data is already in the transactions table (pre-joined).
One join with accounts is needed to get account_currency for currency_match.

In [3]:
# Join to get account_currency for currency_match check
df = df.merge(
    accounts[['account_id', 'account_currency']],
    on='account_id',
    how='left'
)

# currency_match check — zero variance in this dataset (all match), so drop it
# Keeping account_currency temporarily, dropped at assembly step

# Feature 8: rbi_risk_score
# Encode RBI risk category as ordered integer: Low=0, Medium=1, High=2
risk_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['rbi_risk_score'] = df['rbi_risk_category'].map(risk_map)

# Feature 9: days_since_incorporation
# New companies initiating large cross-border payments = higher risk
df['days_since_incorporation'] = (df['business_age_years'] * 365).astype(int)

# Feature 10: turnover_to_txn_ratio
# What fraction of annual revenue is this single transaction?
# Using INR/INR — avoids FX rate dependency (training-serving skew risk)
df['turnover_to_txn_ratio'] = (
    df['amount_inr'] / df['declared_annual_turnover_inr']
).round(4)

# Features 11-15: business_type one-hot encoding
# No natural order exists between business types — one-hot is correct
# EdTech is dropped as reference category (drop_first=True)
business_dummies = pd.get_dummies(df['business_type'], prefix='biz', drop_first=True)
df = pd.concat([df, business_dummies], axis=1)
biz_cols = [c for c in business_dummies.columns]
df[biz_cols] = df[biz_cols].astype(int)

print('Customer/Account features built. Shape:', df.shape)
print('Business type dummy columns:', biz_cols)

Customer/Account features built. Shape: (50800, 79)
Business type dummy columns: ['biz_Exporter', 'biz_IT Services', 'biz_Importer', 'biz_SME', 'biz_SaaS']


## Section 4 — Behavioural Features (6)
**Critical rule:** All rolling windows use `closed='left'` — the current transaction is excluded.
This prevents future leakage. In production, future data does not exist.

In [4]:
# Sort by customer and timestamp — required for all rolling window calculations
df = df.sort_values(['customer_id', 'timestamp']).reset_index(drop=True)
print('Sorted. Shape:', df.shape)

Sorted. Shape: (50800, 79)


In [5]:
# Feature 16: txn_count_24h
# How many transactions has this customer made in the last 24 hours?
# Velocity spikes are a strong fraud indicator.
df = df.set_index('timestamp')
df['txn_count_24h'] = (
    df.groupby('customer_id')['amount_usd']
    .rolling('24h', closed='left')
    .count()
    .reset_index(level=0, drop=True)
    .fillna(0)
    .astype(int)
)
df = df.reset_index()

# Feature 17: txn_count_7d
# Same logic, wider 7-day window — catches slower velocity patterns.
df = df.set_index('timestamp')
df['txn_count_7d'] = (
    df.groupby('customer_id')['amount_usd']
    .rolling('7D', closed='left')
    .count()
    .reset_index(level=0, drop=True)
    .fillna(0)
    .astype(int)
)
df = df.reset_index()
print('Velocity features built.')

Velocity features built.


In [6]:
# Feature 18: amount_zscore_30d
# How many standard deviations is this amount from the customer's 30-day average?
# z-score > 2 has 13.71% fraud rate vs 0.20% baseline.
df = df.set_index('timestamp')
rolling_mean_30d = (
    df.groupby('customer_id')['amount_usd']
    .rolling('30D', closed='left').mean()
    .reset_index(level=0, drop=True)
)
rolling_std_30d = (
    df.groupby('customer_id')['amount_usd']
    .rolling('30D', closed='left').std()
    .reset_index(level=0, drop=True)
)
df['amount_zscore_30d'] = (
    (df['amount_usd'] - rolling_mean_30d) / rolling_std_30d
).fillna(0).replace([np.inf, -np.inf], 0).round(4)
df = df.reset_index()

# Feature 19: amount_zscore_90d
# Wider 90-day baseline — z-score > 2 has 23.89% fraud rate (strongest feature built).
df = df.set_index('timestamp')
rolling_mean_90d = (
    df.groupby('customer_id')['amount_usd']
    .rolling('90D', closed='left').mean()
    .reset_index(level=0, drop=True)
)
rolling_std_90d = (
    df.groupby('customer_id')['amount_usd']
    .rolling('90D', closed='left').std()
    .reset_index(level=0, drop=True)
)
df['amount_zscore_90d'] = (
    (df['amount_usd'] - rolling_mean_90d) / rolling_std_90d
).fillna(0).replace([np.inf, -np.inf], 0).round(4)
df = df.reset_index()
print('Z-score features built.')

Z-score features built.


In [7]:
# Feature 20: days_since_last_txn
# Dormant accounts suddenly reactivating = known fraud pattern.
# -1 = first transaction for this customer (no previous exists).
df['prev_txn_timestamp'] = df.groupby('customer_id')['timestamp'].shift(1)
df['days_since_last_txn'] = (
    (df['timestamp'] - df['prev_txn_timestamp'])
    .dt.total_seconds() / 86400
).round(2).fillna(-1)
df = df.drop(columns=['prev_txn_timestamp'])

# Feature 21: quarter_end_flag
# Indian companies inflate exports at quarter end to hit revenue targets (TBML pattern).
# Quarter-end months: March(3), June(6), September(9), December(12)
QUARTER_END_MONTHS = {3, 6, 9, 12}
df['quarter_end_flag'] = (
    (df['timestamp'].dt.month.isin(QUARTER_END_MONTHS)) &
    (df['timestamp'].dt.day >= 15)
).astype(int)

print('Behavioural features built. Shape:', df.shape)

Behavioural features built. Shape: (50800, 85)


## Section 5 — Risk / Regulatory Features (1)
FATF risk score per destination country based on Financial Action Task Force classification.

In [8]:
# Feature 22: corridor_fatf_score
# 0=Low (FATF member, strong controls)
# 1=Medium (recently removed from grey list, still monitored)
# 2=High (elevated AML risk)
# 3=Very High (FATF blacklist)
FATF_SCORES = {
    'US'        : 0,  # FATF founding member
    'UK'        : 0,  # FATF member
    'Germany'   : 0,  # FATF member
    'Australia' : 0,  # FATF member
    'Canada'    : 0,  # FATF member
    'Singapore' : 0,  # FATF member, strong AML regime
    'UAE'       : 1,  # Removed from grey list 2024, still monitored
    'Malaysia'  : 1,  # Removed from grey list 2023, still monitored
    'China'     : 2,  # Elevated AML risk, opacity concerns
    'Myanmar'   : 3,  # FATF blacklist — 100% fraud rate in dataset
}
# Unknown countries default to medium risk (1)
df['corridor_fatf_score'] = df['beneficiary_country'].map(FATF_SCORES).fillna(1)

print('Risk features built. Shape:', df.shape)
print('\nFraud rate by FATF score:')
print(df.groupby('corridor_fatf_score')['is_fraud'].mean().round(4))

Risk features built. Shape: (50800, 86)

Fraud rate by FATF score:
corridor_fatf_score
0.0    0.0041
1.0    0.0080
2.0    0.0037
3.0    1.0000
Name: is_fraud, dtype: float64


## Section 6 — Assemble Feature Matrix and Save
Drop all raw source columns. Only engineered features and clean binary/numeric columns remain.

In [9]:
COLUMNS_TO_DROP = [
    # Identifiers
    'transaction_id', 'customer_id', 'account_id', 'beneficiary_id',
    # Raw datetime
    'timestamp',
    # Leakage and Not-Applicable NaN columns
    'fraud_type', 'failure_reason', 'reversal_reason',
    # Duplicates — engineered versions exist
    'amount_usd',                    # -> log_amount_usd
    'amount_inr',                    # -> turnover_to_txn_ratio
    'hour_of_day',                   # -> submission_hour
    'day_of_week',                   # -> submission_day
    'business_age_years',            # -> days_since_incorporation
    'declared_annual_turnover_inr',  # -> turnover_to_txn_ratio
    'rbi_risk_category',             # -> rbi_risk_score
    'business_type',                 # -> biz_* dummies
    # Geographic raw — engineered versions exist
    'beneficiary_country',           # -> corridor_fatf_score
    'beneficiary_bank_country',
    'business_city', 'business_state', 'business_region',
    # Payment metadata raw
    'payment_rail',                  # -> rail_corridor_mismatch
    'currency', 'account_currency',
    'fema_purpose_code',             # -> fema_missing_flag
    # Categorical columns not encoded
    'beneficiary_type', 'incorporation_type',
    'employee_count_range', 'annual_revenue_tier',
    'industry_sector', 'account_type',
    'transaction_status', 'transaction_type',
]

# Only drop columns that actually exist in df (avoids KeyError)
existing_to_drop = [c for c in COLUMNS_TO_DROP if c in df.columns]

X = df.drop(columns=existing_to_drop + ['is_fraud'])
y = df['is_fraud']

print('=== FEATURE MATRIX ===')
print('Shape:', X.shape)
print('NaN count:', X.isnull().sum().sum())
print('Fraud cases:', y.sum(), f'({y.mean()*100:.3f}%)')
print('\nFeatures:', X.columns.tolist())

=== FEATURE MATRIX ===
Shape: (50800, 52)
NaN count: 0
Fraud cases: 250 (0.492%)

Features: ['month', 'kyc_verified', 'has_iec', 'is_opc', 'director_count', 'is_msme_registered', 'dispute_count_90d', 'failed_txn_count_30d', 'has_prior_fraud_flag', 'account_age_days', 'is_dormant_account', 'days_since_account_last_used', 'is_primary_account', 'is_new_beneficiary', 'is_new_country', 'counterparty_sanction_flag', 'invoice_match_flag', 'has_supporting_doc', 'is_round_number', 'firc_generated', 'time_since_last_login_hrs', 'is_new_device', 'is_foreign_ip', 'is_failed', 'is_reversed', 'retry_count', 'fema_missing_flag', 'has_invoice', 'login_missing_flag', 'supporting_doc_missing_flag', 'log_amount_usd', 'submission_hour', 'submission_day', 'is_near_threshold', 'is_off_hours', 'is_weekend', 'rail_corridor_mismatch', 'rbi_risk_score', 'days_since_incorporation', 'turnover_to_txn_ratio', 'biz_Exporter', 'biz_IT Services', 'biz_Importer', 'biz_SME', 'biz_SaaS', 'txn_count_24h', 'txn_count_7d', 

In [10]:
os.makedirs('../data/processed', exist_ok=True)

X.to_parquet('../data/processed/feature_matrix.parquet', index=False)
y.to_frame().to_parquet('../data/processed/target.parquet', index=False)

# Verify
X_check = pd.read_parquet('../data/processed/feature_matrix.parquet')
y_check = pd.read_parquet('../data/processed/target.parquet')

print('feature_matrix.parquet saved. Shape:', X_check.shape)
print('target.parquet saved.         Shape:', y_check.shape)
print('Fraud cases in target:        ', y_check['is_fraud'].sum())
print('\nNotebook 04 complete.')

feature_matrix.parquet saved. Shape: (50800, 52)
target.parquet saved.         Shape: (50800, 1)
Fraud cases in target:         250

Notebook 04 complete.
